In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

panel = pd.read_csv('../data/madrid_panel_prox.csv', parse_dates=['fecha'])
panel = panel.sort_values(['id_local', 'fecha']).reset_index(drop=True)

fechas = sorted(panel['fecha'].unique())
dias = pd.Series(pd.to_datetime(fechas)).diff().dt.days.values   # días entre fechas consecutivas

rev = panel.pivot(index='id_local', columns='fecha', values='n_reviews')
sco = panel.pivot(index='id_local', columns='fecha', values='score')
pre = panel.pivot(index='id_local', columns='fecha', values='precio_medio')
estaticas = panel.groupby('id_local')[['categoria', 'n_100m', 'n_250m', 'n_500m',
                                       'n_mismo_tipo_250m', 'dist_vecino_m']].first()
print(rev.shape)

(2029, 18)


In [2]:
filas = []
for k in range(2, len(fechas) - 1):          # k = fecha de origen
    d = pd.DataFrame(index=rev.index)
    d['origen'] = k
    d['fecha_origen'] = fechas[k]
    d['resenas'] = rev.iloc[:, k]
    d['score'] = sco.iloc[:, k]
    d['precio'] = pre.iloc[:, k]
    d['vel_1'] = ((rev.iloc[:, k] - rev.iloc[:, k - 1]) / dias[k]).clip(lower=0) * 30
    d['vel_2'] = ((rev.iloc[:, k] - rev.iloc[:, k - 2]) / (dias[k] + dias[k - 1])).clip(lower=0) * 30
    d['cambio_score'] = sco.iloc[:, k] - sco.iloc[:, k - 2]
    # Lo que queremos predecir: reseñas nuevas cada 30 días hasta la siguiente fecha
    d['objetivo'] = ((rev.iloc[:, k + 1] - rev.iloc[:, k]) / dias[k + 1]).clip(lower=0) * 30
    filas.append(d)

datos = pd.concat(filas).join(estaticas)
datos['categoria'] = datos['categoria'].astype('category')
datos = datos.dropna(subset=['resenas', 'objetivo', 'vel_1', 'vel_2'])

print(datos.shape)
datos['objetivo'].describe().round(2)

(30352, 15)


count    30352.00
mean         5.06
std         35.31
min          0.00
25%          0.00
50%          1.15
75%          3.87
max       2625.00
Name: objetivo, dtype: float64

In [3]:
ORIGEN_TEST = 13
train = datos[datos['origen'] < ORIGEN_TEST]
test = datos[datos['origen'] >= ORIGEN_TEST]
print('Entrenamiento:', len(train), '| fechas origen hasta', train['fecha_origen'].max().date())
print('Test:', len(test), '| fechas origen desde', test['fecha_origen'].min().date())

Entrenamiento: 22241 | fechas origen hasta 2026-03-27
Test: 8111 | fechas origen desde 2026-04-15


In [4]:
y = test['objetivo']
mae_cero = mean_absolute_error(y, np.zeros(len(y)))
mae_v1 = mean_absolute_error(y, test['vel_1'])
mae_v2 = mean_absolute_error(y, test['vel_2'])
mae_media = mean_absolute_error(y, np.full(len(y), train['objetivo'].median()))
print(f'Predecir 0:                     MAE = {mae_cero:.2f}')
print(f'Predecir la mediana del train:  MAE = {mae_media:.2f}')
print(f'Repetir la velocidad anterior:  MAE = {mae_v1:.2f}')
print(f'Repetir la velocidad 2 fechas:  MAE = {mae_v2:.2f}')

Predecir 0:                     MAE = 5.94
Predecir la mediana del train:  MAE = 5.63
Repetir la velocidad anterior:  MAE = 5.60
Repetir la velocidad 2 fechas:  MAE = 4.23


In [5]:
VARIABLES = ['resenas', 'score', 'precio', 'vel_1', 'vel_2', 'cambio_score',
             'categoria', 'n_100m', 'n_250m', 'n_500m', 'n_mismo_tipo_250m', 'dist_vecino_m']

modelo = HistGradientBoostingRegressor(loss='absolute_error', max_iter=200, learning_rate=0.05,
                                       categorical_features='from_dtype', random_state=0)
modelo.fit(train[VARIABLES], train['objetivo'])
pred = modelo.predict(test[VARIABLES]).clip(min=0)

mae_modelo = mean_absolute_error(y, pred)
print(f'Modelo (gradient boosting):     MAE = {mae_modelo:.2f}')
print(f'Mejora frente a repetir vel_2:  {(1 - mae_modelo / mae_v2) * 100:.1f} %')

Modelo (gradient boosting):     MAE = 4.02
Mejora frente a repetir vel_2:  5.0 %


In [6]:
from sklearn.inspection import permutation_importance

muestra = test.sample(min(3000, len(test)), random_state=0)
imp = permutation_importance(modelo, muestra[VARIABLES], muestra['objetivo'],
                             scoring='neg_mean_absolute_error', n_repeats=5, random_state=0)
pd.Series(imp.importances_mean, index=VARIABLES).sort_values(ascending=False).round(3)

vel_2                1.447
resenas              0.656
vel_1                0.039
n_500m               0.033
score                0.013
precio               0.009
dist_vecino_m        0.007
n_100m               0.005
categoria            0.005
n_mismo_tipo_250m    0.003
n_250m               0.003
cambio_score        -0.000
dtype: float64

In [7]:
import joblib

Path('../models').mkdir(exist_ok=True)
joblib.dump({'modelo': modelo, 'variables': VARIABLES}, '../models/modelo_base.joblib')
datos.to_csv('../data/madrid_entrenamiento.csv', index=False)
print('Modelo guardado en ../models/modelo_base.joblib')

Modelo guardado en ../models/modelo_base.joblib
